In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ayomidefagbolade/related-only-data/df_related_deduplicated.csv


In [2]:
#!pip install vllm

In [3]:
#!pip install "protobuf<6.0.0" --force-reinstall

In [4]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

In [5]:
import os
import pandas as pd

In [6]:
import re
import ast

# This is the function you can apply to your DataFrame columns
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Replace literal "\r\n", "\r", or "\n" text with a space
    text = re.sub(r'\\r\\n|\\r|\\n', ' ', text)
    
    # 2. Convert literal backslash-escaped quotes (\') back to normal single quotes (')
    text = re.sub(r"\\'", "'", text)
    
    # 3. Strip any literal single quotes wrapper around the entire text block
    text = text.strip("'")
    
    # 4. Collapse any double/triple spaces down to a single space
    text = re.sub(r'\s+', ' ', text)

    
    return text.strip()


In [7]:
folder_path = "/kaggle/input/datasets/ayomidefagbolade/related-only-data"
csv_files = sorted([f for f in os.listdir(folder_path) if f.endswith('.csv')])
dataframes = {}
for file_name in csv_files:
    file_path = os.path.join(folder_path, file_name)
    # Store it using the file name (without '.csv') as the key
    key = file_name.replace('.csv', '')
    dataframes[key] = pd.read_csv(file_path)
    dataframes[key]['Body_Text'] = dataframes[key]['Body_Text'].apply(clean_text)

  

In [8]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("huggingface")
os.environ["HF_TOKEN"] = secret_value_0

In [9]:
# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")

In [10]:
import random
# 1. Clear out previous zombie processes from your crash
!pkill -f vllm

# 2. FORCE vLLM back to the stable V0 architecture (Fixes Kaggle Multi-GPU)
os.environ["VLLM_VERSION"] = "v0"

# 3. Apply standard multi-GPU network overrides
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["VLLM_HOST_IP"] = "127.0.0.1"
os.environ["GLOO_SOCKET_IFNAME"] = "lo"
os.environ["NCCL_SOCKET_IFNAME"] = "lo"
os.environ["MASTER_PORT"] = str(random.randint(20000, 30000))

In [12]:
llm = LLM(model="meta-llama/Llama-3.1-8B-Instruct",
         dtype="half",
          tensor_parallel_size=2,
          gpu_memory_utilization=0.85, 
          disable_custom_all_reduce=True,
          max_model_len=10000,   
          
    )

INFO 07-19 02:33:36 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 10000, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'disable_custom_all_reduce': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
WARNING 07-19 02:33:36 [envs.py:2041] Unknown vLLM environment variable detected: VLLM_VERSION
INFO 07-19 02:33:36 [model.py:619] Resolved architecture: LlamaForCausalLM
WARNING 07-19 02:33:36 [model.py:2114] Casting torch.bfloat16 to torch.float16.
INFO 07-19 02:33:36 [model.py:1776] Using max model len 10000
INFO 07-19 02:33:36 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=21069) INFO 07-19 02:33:39 [core.py:114] Initializing a V1 LLM engine (v0.25.1) with config: model='meta-llama/Llama-3.1-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.1-8B-Instruct', skip_tokenizer_init=False, tokenizer_mod

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(Worker_TP0 pid=21087) INFO 07-19 02:34:22 [default_loader.py:430] Loading weights took 32.92 seconds
(Worker_TP0 pid=21087) INFO 07-19 02:34:23 [model_runner.py:302] Model loading took 7.54 GiB and 35.737309 seconds
(Worker_TP0 pid=21087) WARNING 07-19 02:34:23 [topk_topp_sampler.py:62] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(Worker_TP1 pid=21088) INFO 07-19 02:34:29 [model_runner.py:302] Model loading took 7.54 GiB and 41.643312 seconds
(Worker_TP0 pid=21087) INFO 07-19 02:34:40 [backends.py:1089] Using cache directory: /root/.cache/vllm/torch_compile_cache/58fc514c92/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=21087) INFO 07-19 02:34:40 [backends.py:1148] Dynamo bytecode transform time: 10.17 s


(Worker_TP0 pid=21087) [rank0]:W0719 02:34:43.118000 21087 torch/_inductor/utils.py:1731] Not enough SMs to use max_autotune_gemm mode
(Worker_TP1 pid=21088) [rank1]:W0719 02:34:43.159000 21088 torch/_inductor/utils.py:1731] Not enough SMs to use max_autotune_gemm mode


(Worker_TP0 pid=21087) INFO 07-19 02:34:49 [backends.py:378] Cache the graph of compile range (1, 8192) for later use
(Worker_TP0 pid=21087) INFO 07-19 02:34:56 [backends.py:393] Compiling a graph for compile range (1, 8192) takes 15.76 s
(Worker_TP0 pid=21087) INFO 07-19 02:35:00 [decorators.py:708] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/c461689aed37bad072411b543c06a2feca49d6cb21da5c7e7fbf5238a30a86b4/rank_0_0/model
(Worker_TP0 pid=21087) INFO 07-19 02:35:00 [monitor.py:53] torch.compile took 30.30 s in total
(Worker_TP0 pid=21087) INFO 07-19 02:35:00 [monitor.py:81] Initial profiling/warmup run took 0.44 s
(Worker_TP0 pid=21087) INFO 07-19 02:35:06 [gpu_worker.py:538] Available KV cache memory: 4.26 GiB
(EngineCore pid=21069) INFO 07-19 02:35:06 [kv_cache_utils.py:2146] GPU KV cache size: 69,760 tokens
(EngineCore pid=21069) INFO 07-19 02:35:06 [kv_cache_utils.py:2147] Maximum concurrency for 10,000 tokens per request: 6.98x
(Worker_TP1

Capturing CUDA graphs (FULL):  97%|█████████▋| 34/35 [00:08<00:00,  3.69it/s]

(Worker_TP1 pid=21088) INFO 07-19 02:35:25 [model_runner.py:722] Graph capturing finished in 19 secs, took 1.22 GiB


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:10<00:00,  3.44it/s]


(Worker_TP0 pid=21087) INFO 07-19 02:35:25 [model_runner.py:722] Graph capturing finished in 19 secs, took 1.22 GiB
(Worker_TP0 pid=21087) INFO 07-19 02:35:43 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(Worker_TP1 pid=21088) INFO 07-19 02:35:43 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=21069) INFO 07-19 02:35:43 [core.py:337] init engine (profile, create kv cache, warmup model) took 74.58 s (compilation: 30.30 s)
(EngineCore pid=21069) INFO 07-19 02:35:45 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(Worker_TP0 pid=21087) WARNING 07-19 02:36:33 [jit_monitor.py:129] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


In [13]:
sampling_params = SamplingParams(
    temperature=0.01, repetition_penalty=1.0, max_tokens=512,
skip_special_tokens=True, seed=42, )
# Prepare your prompts

In [14]:
import os
import re
import json
import time
import random
from collections import defaultdict

import pandas as pd  # only used by run_extraction() to reload checkpointed CSVs

# ============ SWITCHES ============
TEST_MODE = True   # set to False when you're happy with results
TEST_MODE = False
TEST_SAMPLE_SIZE = 50

# Full-run stage switches — run ONE of these per Kaggle session.
# e.g. session 1: DO_CLASSIFICATION=True,  DO_EXTRACTION=False
#      session 2: DO_CLASSIFICATION=False, DO_EXTRACTION=True
DO_CLASSIFICATION = False
DO_EXTRACTION = True

# Resume point for classification. If set (e.g. 2015), any dataset whose
# key's year is < START_YEAR is skipped entirely for this run — including
# tokenizing/prompt-building, not just generation. Leave as None to process
# everything. Datasets already fully classified in a previous session don't
# need to be re-included; just bump START_YEAR to the next year you need.
START_YEAR = None

# Where the extraction stage reads its "{df_key}_classified.csv" checkpoint
# files from. Leave as "." if extraction runs in the same session/working
# directory as classification. If you're resuming extraction in a fresh
# Kaggle session, set this to wherever those CSVs live — e.g. a Kaggle
# dataset you attached as input:
#   EXTRACTION_INPUT_DIR = "/kaggle/input/my-classified-articles"
EXTRACTION_INPUT_DIR = "/kaggle/input/datasets/ayomidefagbolade/related-only-data/df_related_deduplicated.csv"

# Where extraction writes its updated CSVs back out. /kaggle/input is
# read-only, so if EXTRACTION_INPUT_DIR points there, this MUST point
# somewhere writable instead, e.g. "/kaggle/working".
EXTRACTION_OUTPUT_DIR = "."
# ===================================

CHUNK_SIZE = 500
LONG_CHUNK_SIZE = 20          # small batch to prevent Out-Of-Memory errors on long articles
LONG_ARTICLE_THRESHOLD = 25000   # articles above this token count go through the isolated queue
MAX_ALLOWED_TOKENS = 32000       # hard cutoff (model ceiling is 32768) — truncate anything past this

# ---------------- PROMPTS ----------------

system_prompt = """You are an expert content classifier specializing in agriculture, food systems, and nutrition, with a specific research focus on staple food crops: cassava, yam, banana/plantain, sorghum, and sweet potato.

Your task is to analyze the provided news article and determine whether its core topic is substantially about one or more of these five focus crops as agricultural or food-security subjects — their cultivation, production, supply chains, trade, food security role, or nutritional/health impact at a population or policy level.

Strictly classify the article into one of the following two categories:
- RELATED
- NOT RELATED

CRITICAL CLASSIFICATION BOUNDARIES:
1. Classify as "RELATED" only if the article's core topic substantially concerns cassava, yam, banana/plantain, sorghum, or sweet potato as agricultural, economic, or food-security subjects — including crop production, farming economics, supply chains, trade, food security policy, or nutritional/health impact at a population level.
2. Classify as "NOT RELATED" if the article is about agriculture, food systems, or nutrition generally, but does NOT substantially focus on one of the five listed crops. General farming, other crops, livestock, or unrelated food topics do not qualify, even if broadly agricultural.
3. Classify as "NOT RELATED" if the text is primarily a consumer shopping guide, wholesale market price directory, retail advertisement, or general lifestyle guide on where to buy consumer goods — even if it mentions one of the five crops in passing.
4. Classify as "NOT RELATED" if a focus crop appears only as a FOOD/MEAL/DISH INGREDIENT — for example, school feeding menus, restaurant listings, recipes, meal schedules, or descriptions of what was served or eaten. Mentioning that "yam and egg sauce" was served on a given day is a meal description, not a discussion of yam as a crop, and does NOT qualify as RELATED — regardless of how many times the crop name appears in that context.
5. A brief or incidental mention of a focus crop is NOT sufficient for "RELATED" — the crop must be a substantial subject of the article as an agricultural/food-system topic, not just an ingredient mentioned in passing.

Provide your response in this exact JSON format without any introductory or concluding text:
{
  "classification": "YOUR_CLASSIFICATION",
  "confidence": 0.00,
  "reason": "A brief explanation naming which focus crop (if any) drove the classification, whether it was substantial or incidental, and whether it appeared as an agricultural/food-system subject or merely as a meal ingredient."
}"""

extraction_system_prompt = """You are an expert research assistant analyzing news articles about staple food crops for a food security research project.

You are looking for discussion of these five focus crops: cassava, yam, banana/plantain, sorghum, sweet potato — as agricultural, economic, or food-security subjects (production, farming, supply chains, trade, food security policy, or nutritional/health impact at a population level).

Your task is to identify which of these crops, if any, are substantially discussed in this way, and for each one, extract the specific issue(s) or topic(s) raised about it — in your own concise words, drawn directly from the article's content. Do not use a fixed category system; describe each issue naturally and specifically (e.g. "declining yields due to prolonged drought in the region" rather than just "climate").

Do not assume any of the five crops are discussed — verify this yourself from the article text.

IMPORTANT EXCLUSION: Do NOT extract an "issue" from a crop that appears only as a FOOD/MEAL/DISH INGREDIENT — for example, school feeding menus, restaurant listings, recipes, meal schedules, or descriptions of what was served or eaten. A sentence like "yam and egg sauce was served on Mondays" is a meal description, not an agricultural or food-security issue, and must be excluded even if the crop name is mentioned. Only extract an issue if the article discusses the crop as a subject of production, economics, policy, trade, or population-level nutrition — not as an item on a plate.

If a crop is mentioned only in passing, or only as a meal ingredient with no substantive agricultural/food-security issue attached, do not include it. If none of the five crops are substantially discussed in a qualifying way, return an empty list.

For each issue you extract, also classify its SENTIMENT — the tone with which the article discusses that specific issue for that crop — as one of:
- "positive": the article frames this issue favorably (e.g. bumper harvest, rising prices benefiting farmers, successful intervention, growth in exports, new funding secured)
- "negative": the article frames this issue unfavorably (e.g. crop disease, drought losses, price collapse, food insecurity, supply shortages, policy failures)
- "neutral": the article reports the issue factually/descriptively without a clear favorable or unfavorable framing (e.g. a routine statistic, a general description, or a mixed/balanced account with no dominant tone)

Base the sentiment strictly on how the article itself frames that issue — not on your own opinion of whether the underlying fact is good or bad.

Provide your response in this exact JSON format without any introductory or concluding text:
{
  "crops_discussed": [
    {
      "crop": "CROP_NAME",
      "issues": [
        {"issue": "issue description 1", "sentiment": "positive|negative|neutral"},
        {"issue": "issue description 2", "sentiment": "positive|negative|neutral"}
      ]
    }
  ]
}

If no focus crop has a substantive issue discussed, return:
{
  "crops_discussed": []
}
"""

# ---------------- HELPERS ----------------

def parse_json_response(generated_text, fallback):
    match = re.search(r"\{.*\}", generated_text, re.DOTALL)
    if not match:
        return fallback
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return fallback

def parse_classification(generated_text):
    return parse_json_response(
        generated_text,
        {"classification": "PARSE_ERROR", "confidence": 0.0, "reason": "No/invalid JSON"}
    )

def parse_extraction(generated_text):
    return parse_json_response(generated_text, {"crops_discussed": []})

def build_classification_prompt(article_text):
    prompt = f"""Please classify the following news article:
---
ARTICLE TEXT:
{article_text}
---
Your JSON classification response:"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_extraction_prompt(article_text):
    prompt = f"""Please extract the crop-specific issues discussed in the following article:
---
ARTICLE TEXT:
{article_text}
---
Your JSON response:"""
    messages = [
        {"role": "system", "content": extraction_system_prompt},
        {"role": "user", "content": prompt}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def year_of(df_key):
    """Extract a 4-digit year from a dataset key. Returns None if none found.
    Adjust this if your df_keys don't embed the year as e.g. 'news_2018' or 2018."""
    m = re.search(r"(19|20)\d{2}", str(df_key))
    return int(m.group(0)) if m else None

# ==================================================================
# TEST MODE — quick end-to-end sanity check on a small random sample
# (both stages run together here since it's small/fast, not timeout-prone)
# ==================================================================
if TEST_MODE:
    texts = []
    index_map = []
    for df_key, df in dataframes.items():
        df['Body_Text'] = df['Body_Text'].fillna("")
        for idx, text in df['Body_Text'].items():
            texts.append(text)
            index_map.append((df_key, idx))

    total = len(texts)
    print(f"Total articles available: {total}")

    print("Tokenizing all articles (batched)...")
    t0 = time.time()
    token_id_lists = tokenizer(texts, add_special_tokens=False)['input_ids']
    token_counts = [len(t) for t in token_id_lists]
    print(f"  Done in {time.time() - t0:.1f}s")

    all_prompts = []
    for i, (text, token_count, tokens) in enumerate(zip(texts, token_counts, token_id_lists)):
        if token_count > MAX_ALLOWED_TOKENS:
            df_key, row_idx = index_map[i]
            text = tokenizer.decode(tokens[:MAX_ALLOWED_TOKENS], skip_special_tokens=True)
            print(f"⚠️ Truncated outlier article at [{df_key} | row {row_idx}] "
                  f"from {token_count} down to {MAX_ALLOWED_TOKENS} tokens.")
        all_prompts.append(build_classification_prompt(text))

    random.seed(42)
    sample_idx = random.sample(range(total), min(TEST_SAMPLE_SIZE, total))
    sample_prompts = [all_prompts[i] for i in sample_idx]
    sample_index_map = [index_map[i] for i in sample_idx]

    print(f"\n--- TEST MODE: classifying {len(sample_prompts)} random articles ---\n")
    t0 = time.time()
    outputs = llm.generate(sample_prompts, sampling_params)
    elapsed = time.time() - t0

    classification_results = {}  # (df_key, row_idx) -> result dict

    for (df_key, row_idx), output in zip(sample_index_map, outputs):
        generated_text = output.outputs[0].text
        result = parse_classification(generated_text)
        classification_results[(df_key, row_idx)] = result

        article_preview = dataframes[df_key].at[row_idx, 'Body_Text'][:150]
        print(f"[{df_key} | row {row_idx}]")
        print(f"  Article preview: {article_preview}...")
        print(f"  Classification : {result.get('classification')}")
        print(f"  Confidence     : {result.get('confidence')}")
        print(f"  Reason         : {result.get('reason')}")
        print("-" * 80)

    per_article = elapsed / len(sample_prompts)
    est_total_seconds = per_article * total
    print(f"\n--- Classification timing ---")
    print(f"  {elapsed:.1f}s for {len(sample_prompts)} articles ({per_article:.2f}s/article)")
    print(f"  Estimated full run: {est_total_seconds/60:.1f} min ({est_total_seconds/3600:.2f} hr)")

    related_keys = [k for k, r in classification_results.items() if r.get('classification') == 'RELATED']

    if related_keys:
        print(f"\n--- TEST MODE: extracting issues for {len(related_keys)} RELATED articles ---\n")
        extraction_prompts = [
            build_extraction_prompt(dataframes[df_key].at[row_idx, 'Body_Text'])
            for (df_key, row_idx) in related_keys
        ]

        t0 = time.time()
        extraction_outputs = llm.generate(extraction_prompts, sampling_params)
        elapsed_ext = time.time() - t0

        for (df_key, row_idx), output in zip(related_keys, extraction_outputs):
            generated_text = output.outputs[0].text
            ext_result = parse_extraction(generated_text)
            crops_found = ext_result.get('crops_discussed', [])

            print(f"[{df_key} | row {row_idx}]")
            if crops_found:
                for entry in crops_found:
                    print(f"  Crop: {entry.get('crop')}")
                    for issue_entry in entry.get('issues', []):
                        issue_text = issue_entry.get('issue') if isinstance(issue_entry, dict) else issue_entry
                        sentiment = issue_entry.get('sentiment') if isinstance(issue_entry, dict) else None
                        if sentiment:
                            print(f"    - [{sentiment}] {issue_text}")
                        else:
                            print(f"    - {issue_text}")
            else:
                print(f"  No crops confirmed (possible classification false positive)")
            print("-" * 80)

        per_article_ext = elapsed_ext / len(related_keys)
        print(f"\n--- Extraction timing ---")
        print(f"  {elapsed_ext:.1f}s for {len(related_keys)} articles ({per_article_ext:.2f}s/article)")
    else:
        print("\nNo RELATED articles in this sample to run extraction on.")

    print("\nTEST MODE complete. No CSVs written, no dataframe columns modified.")
    print("Set TEST_MODE = False to run the full pipeline.")

# ==================================================================
# FULL RUN — classification and extraction are separate calls below.
# Set DO_CLASSIFICATION / DO_EXTRACTION at the top to control which
# one runs in this session.
# ==================================================================
else:

    # -------------------- STAGE 1: CLASSIFICATION --------------------
    def run_classification():
        for df in dataframes.values():
            if 'classification' not in df.columns:
                df['classification'] = None
                df['confidence'] = None
                df['reason'] = None

        # Build texts/index_map only for datasets in scope for this run.
        texts = []
        index_map = []  # (df_key, row_index)

        for df_key, df in dataframes.items():
            if START_YEAR is not None:
                y = year_of(df_key)
                if y is not None and y < START_YEAR:
                    print(f"Skipping {df_key} (year {y} < START_YEAR {START_YEAR})")
                    continue
            df['Body_Text'] = df['Body_Text'].fillna("")
            for idx, text in df['Body_Text'].items():
                texts.append(text)
                index_map.append((df_key, idx))

        total = len(texts)
        print(f"Total articles in scope for this classification run: {total}")
        if total == 0:
            print("Nothing to classify — check START_YEAR / your dataset keys.")
            return

        print("Tokenizing articles in scope (batched)...")
        t0 = time.time()
        token_id_lists = tokenizer(texts, add_special_tokens=False)['input_ids']
        token_counts = [len(t) for t in token_id_lists]
        print(f"  Done in {time.time() - t0:.1f}s")

        all_prompts = []
        is_long = []
        for i, (text, token_count, tokens) in enumerate(zip(texts, token_counts, token_id_lists)):
            if token_count > MAX_ALLOWED_TOKENS:
                df_key, row_idx = index_map[i]
                text = tokenizer.decode(tokens[:MAX_ALLOWED_TOKENS], skip_special_tokens=True)
                print(f"⚠️ Truncated outlier article at [{df_key} | row {row_idx}] "
                      f"from {token_count} down to {MAX_ALLOWED_TOKENS} tokens.")
            all_prompts.append(build_classification_prompt(text))
            is_long.append(token_count > LONG_ARTICLE_THRESHOLD)

        standard_prompts = []
        standard_index_map = []
        long_prompts = []
        long_index_map = []

        for prompt, (df_key, idx), long_flag in zip(all_prompts, index_map, is_long):
            if long_flag:
                long_prompts.append(prompt)
                long_index_map.append((df_key, idx))
            else:
                standard_prompts.append(prompt)
                standard_index_map.append((df_key, idx))

        print(f"Total Standard Articles: {len(standard_prompts)}")
        print(f"Total Extra Long Articles (Isolated): {len(long_prompts)}")

        grouped_standard_prompts = defaultdict(list)
        grouped_standard_indices = defaultdict(list)
        for prompt, (df_key, idx) in zip(standard_prompts, standard_index_map):
            grouped_standard_prompts[df_key].append(prompt)
            grouped_standard_indices[df_key].append(idx)

        # --- Stage 1a: classify standard articles, per df_key, chunked ---
        for df_key in grouped_standard_prompts:
            df = dataframes[df_key]
            print(f"\n==== Starting standard-queue classification for dataset: {df_key} ====")

            df_prompts = grouped_standard_prompts[df_key]
            df_indices = grouped_standard_indices[df_key]

            year_total = len(df_prompts)
            num_chunks = (year_total + CHUNK_SIZE - 1) // CHUNK_SIZE

            for chunk_num in range(num_chunks):
                start = chunk_num * CHUNK_SIZE
                end = min(start + CHUNK_SIZE, year_total)

                chunk_prompts = df_prompts[start:end]
                chunk_idxs = df_indices[start:end]

                t0 = time.time()
                outputs = llm.generate(chunk_prompts, sampling_params)
                elapsed = time.time() - t0

                for idx, output in zip(chunk_idxs, outputs):
                    generated_text = output.outputs[0].text
                    result = parse_classification(generated_text)

                    df.at[idx, 'classification'] = result.get('classification', 'PARSE_ERROR')
                    df.at[idx, 'confidence'] = result.get('confidence', 0.0)
                    df.at[idx, 'reason'] = result.get('reason', '')

                print(f"[{df_key}] Chunk {chunk_num + 1}/{num_chunks} done "
                      f"({end}/{year_total} articles, {elapsed:.1f}s)")

                df.to_csv(f"{df_key}_classified.csv", index=False)

            print(f"==== Finished standard queue for {df_key}! "
                  f"Metrics so far: {df['classification'].value_counts().to_dict()} ====")

        # --- Stage 1b: classify extra-long articles, isolated small-batch queue ---
        num_long_chunks = (len(long_prompts) + LONG_CHUNK_SIZE - 1) // LONG_CHUNK_SIZE
        print(f"\n--- Starting isolated execution for {len(long_prompts)} extra long articles ---")

        for chunk_num in range(num_long_chunks):
            start = chunk_num * LONG_CHUNK_SIZE
            end = min(start + LONG_CHUNK_SIZE, len(long_prompts))

            chunk_prompts = long_prompts[start:end]
            chunk_index_map = long_index_map[start:end]

            outputs = llm.generate(chunk_prompts, sampling_params)

            for (df_key, row_idx), output in zip(chunk_index_map, outputs):
                generated_text = output.outputs[0].text
                result = parse_classification(generated_text)

                dataframes[df_key].at[row_idx, 'classification'] = result.get('classification', 'PARSE_ERROR')
                dataframes[df_key].at[row_idx, 'confidence'] = result.get('confidence', 0.0)
                dataframes[df_key].at[row_idx, 'reason'] = result.get('reason', '')

            for touched_key in set(k for k, _ in chunk_index_map):
                dataframes[touched_key].to_csv(f"{touched_key}_classified.csv", index=False)

            print(f"[Long Queue] Chunk {chunk_num + 1}/{num_long_chunks} done.")

        print("\nClassification run complete for this scope.")
        for df_key in (list(grouped_standard_prompts.keys()) + [k for k, _ in long_index_map]):
            df = dataframes[df_key]
            print(f"{df_key}: {df['classification'].value_counts().to_dict()}")

    # -------------------- STAGE 2: EXTRACTION (separate call) --------------------
    def run_extraction():
        """Reads the checkpointed *_classified.csv files from disk (so this works
        even in a fresh Kaggle session after classification finished separately),
        finds RELATED rows that don't have crops_discussed yet, and extracts issues."""
        print("\n======================= EXTRACTION STAGE =======================")
        print(f"Reading classified CSVs from: {os.path.abspath(EXTRACTION_INPUT_DIR)}")
        print(f"Writing updated CSVs to:      {os.path.abspath(EXTRACTION_OUTPUT_DIR)}")
        os.makedirs(EXTRACTION_OUTPUT_DIR, exist_ok=True)

        extraction_index_map = []
        extraction_prompts = []

        for df_key in list(dataframes.keys()):
            csv_path = os.path.join(EXTRACTION_INPUT_DIR, f"{df_key}_classified.csv")
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                dataframes[df_key] = df
            else:
                df = dataframes[df_key]
                if 'classification' not in df.columns:
                    print(f"⚠️ No classification found for {df_key} — looked for {csv_path}, "
                          f"and no in-memory results either — skipping.")
                    continue

            if 'crops_discussed' not in df.columns:
                df['crops_discussed'] = None

            df['Body_Text'] = df['Body_Text'].fillna("")

            already_done_mask = df['crops_discussed'].notna()
            related_mask = (df['classification'] == 'RELATED') & (~already_done_mask)
            related_rows = df[related_mask]

            skipped = int(((df['classification'] == 'RELATED') & already_done_mask).sum())
            if skipped:
                print(f"[{df_key}] Skipping {skipped} already-extracted RELATED rows.")

            for idx, text in related_rows['Body_Text'].items():
                extraction_prompts.append(build_extraction_prompt(text))
                extraction_index_map.append((df_key, idx))

        ext_total = len(extraction_prompts)
        print(f"\nTotal RELATED articles to extract issues from (this run): {ext_total}")
        if ext_total == 0:
            print("Nothing left to extract.")
            return

        ext_num_chunks = (ext_total + CHUNK_SIZE - 1) // CHUNK_SIZE

        for chunk_num in range(ext_num_chunks):
            start = chunk_num * CHUNK_SIZE
            end = min(start + CHUNK_SIZE, ext_total)

            chunk_prompts = extraction_prompts[start:end]
            chunk_index_map = extraction_index_map[start:end]

            t0 = time.time()
            outputs = llm.generate(chunk_prompts, sampling_params)
            elapsed = time.time() - t0

            for (df_key, row_idx), output in zip(chunk_index_map, outputs):
                generated_text = output.outputs[0].text
                ext_result = parse_extraction(generated_text)
                dataframes[df_key].at[row_idx, 'crops_discussed'] = json.dumps(ext_result.get('crops_discussed', []))

            print(f"[Extract] Chunk {chunk_num + 1}/{ext_num_chunks} done "
                  f"({end}/{ext_total} articles, {elapsed:.1f}s) — checkpointing...")

            for touched_key in set(k for k, _ in chunk_index_map):
                out_path = os.path.join(EXTRACTION_OUTPUT_DIR, f"{touched_key}_classified.csv")
                dataframes[touched_key].to_csv(out_path, index=False)

        print("\nExtraction run complete.")

    if DO_CLASSIFICATION:
        run_classification()

    if DO_EXTRACTION:
        run_extraction()

    if not DO_CLASSIFICATION and not DO_EXTRACTION:
        print("Both DO_CLASSIFICATION and DO_EXTRACTION are False — nothing to do. "
              "Set one of them to True at the top of the script.")


======================= EXTRACTION STAGE =======================
Reading classified CSVs from: /kaggle/input/datasets/ayomidefagbolade/related-only-data/df_related_deduplicated.csv
Writing updated CSVs to:      /kaggle/working

Total RELATED articles to extract issues from (this run): 6900


Rendering prompts:   0%|          | 0/500 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 500/500 [09:28<00:00,  1.14s/it, est. speed input: 1137.86 toks/s, output: 156.37 toks/s]


[Extract] Chunk 1/14 done (500/6900 articles, 572.3s) — checkpointing...


Rendering prompts:   0%|          | 0/500 [00:00<?, ?it/s]

...

Processed prompts: 100%|██████████| 500/500 [12:47<00:00,  1.54s/it, est. speed input: 986.77 toks/s, output: 123.19 toks/s]


[Extract] Chunk 13/14 done (6500/6900 articles, 770.7s) — checkpointing...


Rendering prompts:   0%|          | 0/400 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 400/400 [11:02<00:00,  1.66s/it, est. speed input: 937.47 toks/s, output: 121.03 toks/s]


[Extract] Chunk 14/14 done (6900/6900 articles, 665.6s) — checkpointing...

Extraction run complete.
